In [ ]:
import os
import hashlib
import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import torch.utils.data as data
import torchvision.models as models
from PIL import Image
from torch import nn
from tqdm import tqdm


def plot(train, test=None):
    plt.figure(figsize=(8, 6))
    epochs = range(1, len(train) + 1)
    plt.plot(epochs, train, label='Train', color='b')
    if test is not None:
        plt.plot(epochs, test, label='Validation', color='r')
    plt.xlim(1, len(epochs))
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()


def get_hash(image):
    md5 = hashlib.md5()
    md5.update(np.array(image).tobytes())
    return md5.hexdigest()


def load_data(path):
    train_data = pd.read_csv(os.path.join(path, r'train.csv'))
    if False:
        for i, img in enumerate(train_data['image']):
            image = Image.open(os.path.join(path, img))
            train_data.loc[i, 'hash'] = get_hash(image)

        dp = train_data['hash'].value_counts().reset_index()
        dp.columns = ['hash', 'dup_number']
        dp = dp[dp['dup_number'] > 1]
        dp = train_data.merge(dp, on='hash')
        print(dp.head(10))

        duplicate_hashes = train_data[train_data.duplicated(subset=['hash'], keep=False)]
        hashes_to_remove = duplicate_hashes[duplicate_hashes.duplicated(subset=['hash', 'label'], keep=False) == False][
            'hash'].unique()
        train_data = train_data[~train_data['hash'].isin(hashes_to_remove)]

        train_data = train_data.drop_duplicates(subset=['hash', 'label'], keep='first')
    train_img = train_data['image'].values
    train_y, label = pd.factorize(train_data['label'])
    test_data = pd.read_csv(os.path.join(path, r'test.csv'))
    test_img = test_data['image'].values
    return label, train_img, train_y, test_img


class MyDataSet(data.Dataset):
    def __init__(self, root_dir, img, y=None):
        self.root_dir = root_dir
        self.img = img
        if y is None:
            self.y = self.img
            self.trans = norm_trans()
        else:
            self.y = y
            self.trans = get_trans()

    def __getitem__(self, item):
        img = Image.open(os.path.join(self.root_dir, self.img[item]))
        img = np.array(img)
        img = self.trans(image=img)['image']
        return img.transpose((2, 0, 1)), self.y[item]

    def __len__(self):
        return len(self.img)


def eval(dataset, net, loss):
    loss_sum = 0
    acc_sum = 0
    for img, y in data.DataLoader(dataset, eval_batch_size, num_workers=num_workers):
        img = img.to(device)
        y = y.to(device)
        o = net(img)
        l = loss(o, y)
        loss_sum += l.item()
        acc_sum += (torch.argmax(o, 1) == y).sum().item()
    return loss_sum / (len(dataset) + 1e-6), acc_sum / (len(dataset) + 1e-6)


def train():
    dataset = MyDataSet(data_path, train_img, train_y)
    train_data, valid_data = data.random_split(dataset, lengths=[int(len(dataset) * train_ratio),
                                                                 len(dataset) - int(len(dataset) * train_ratio)])
    net, loss, optimizer, scheduler = get_net()
    train_loss_ls = []
    train_acc_ls = []
    valid_loss_ls = []
    valid_acc_ls = []
    for epoch in range(epochs):
        net.train()
        for img, y in data.DataLoader(train_data, batch_size, shuffle=True, num_workers=num_workers):
            img = img.to(device)
            y = y.to(device)
            optimizer.zero_grad()
            o = net(img)
            l = loss(o, y) / batch_size
            l.backward()
            optimizer.step()
        scheduler.step()
        net.eval()
        with torch.no_grad():
            l, a = eval(train_data, net, loss)
            train_loss_ls.append(l)
            train_acc_ls.append(a)
            l, a = eval(valid_data, net, loss)
            valid_loss_ls.append(l)
            valid_acc_ls.append(a)
        print(f'epoch {epoch:02d} train loss {train_loss_ls[-1]:.4f} acc {train_acc_ls[-1]:.4f}, '
              f'valid loss {valid_loss_ls[-1]:.4f} acc {valid_acc_ls[-1]:.4f}')
    plot(train_loss_ls, valid_loss_ls)
    plot(train_acc_ls, valid_acc_ls)
    return net


def predict(net):
    image = []
    res = []
    dataset = MyDataSet(data_path, test_img)
    net.eval()
    with torch.no_grad():
        for img, n in tqdm(data.DataLoader(dataset, eval_batch_size, num_workers=num_workers)):
            img = img.to(device)
            o = net(img)
            image.append(n)
            res.append(label[torch.argmax(o, dim=1).cpu()])
    image = np.concatenate(image, axis=0)
    res = np.concatenate(res, axis=0)
    return pd.DataFrame({
        'image': list(image),
        'label': list(res)
    })


def get_trans():
    return A.Compose([
        A.Resize(360, 360),
        A.RandomResizedCrop(size=(224, 224), scale=(0.64, 1.0)),
        A.OneOf([
            A.Compose([A.HorizontalFlip(p=0.5)]),
            A.Compose([A.Transpose(p=1), A.VerticalFlip(p=0.5)]),
            A.Compose([A.VerticalFlip(p=1), A.HorizontalFlip(p=0.5)]),
            A.Compose([A.Transpose(p=1), A.HorizontalFlip(p=1), A.VerticalFlip(p=0.5)])
        ], p=1),
        A.ShiftScaleRotate(shift_limit=0.25, scale_limit=0.1, rotate_limit=0, border_mode=4),
        A.GaussNoise(var_limit=(1, 75)),
        A.RandomBrightnessContrast(brightness_limit=0.1),
        norm_trans()
    ])


def norm_trans():
    return A.Normalize(
        [0.485, 0.456, 0.406], [0.229, 0.224, 0.225],
        max_pixel_value=255.0, always_apply=True
    )


def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm1d):
        nn.init.ones_(m.weight)
        nn.init.zeros_(m.bias)


def get_net():
    if pretrained:
        net = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    else:
        net = models.resnet50(num_classes=176)
    net.fc = nn.Sequential(
        nn.Linear(net.fc.in_features, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(p=0.5),
        nn.Linear(512, 176)
    )
    net.fc.apply(init_weights)
    net = net.to(device)
    loss = nn.CrossEntropyLoss(reduction='sum').to(device)
    optimizer = optim.AdamW(
        [{'params': [param for name, param in net.named_parameters() if 'fc.' not in name]},
         {'params': net.fc.parameters(), 'lr': lr * 10}],
        lr=lr, weight_decay=weight_decay
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
    return net, loss, optimizer, scheduler


if __name__ == '__main__':
    np.random.seed(144)
    torch.manual_seed(144)
    data_path = '/kaggle/input/classify-leaves' if torch.cuda.is_available() else './'
    device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
    print('Device:', device)
    pretrained = torch.cuda.is_available()
    train_ratio = 0.9
    num_workers = 2
    lr = 0.0001
    weight_decay = 0.01
    batch_size = 24
    eval_batch_size = 128
    epochs = 40
    label, train_img, train_y, test_img = load_data(data_path)
    net = train()
    # torch.save(net.state_dict(), 'model/model.pth')
    result = predict(net)
    result.to_csv('submission.csv', index=False)
